# Violation Heatmap — Viewing "Which Nonlinear Constraint Relaxations are Loose" at a Glance

Nonlinear constraints in MINLP each have different types of convex relaxations (McCormick, power outer approximation, exp convex envelope, etc.), and their looseness varies. `minlpkit.collectors.violation` measures how far each constraint type deviates from the "equation it is supposed to satisfy" by **substituting the root LP relaxation solution into the true nonlinear constraints**.
This dives deeper into the collector—which was touched upon for just one model in district_heating_detailed_physics—using a **model with many constraint types** (reaction kinetics, conversion rate, demand satisfaction, heat removal capacity... 6 types).

1. **What to Observe** — How violation amounts are measured via `getNlRowSolFeasibility`
2. **Visualization** — Heatmap of constraint types × entities + ranking by type
3. **Interpretation** — Identifying the dominant bottleneck and its correspondence with the `weak_relaxation` diagnostic rule
4. **Summary**

The subject is **Batch Reactor Scheduling** (`samples/others/scheduling_plant.py`).
It contains a rich variety of nonlinearities: reaction kinetics (Arrhenius: $\exp(1/T)$), conversion rate (nested $\exp$), demand satisfaction ($n\cdot s\cdot X$ triple product), heating time (bilinear), and heat removal constraints (equivalent to a triple product).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/others"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import scheduling_plant as sp
from minlpkit.collectors.violation import collect_root_violations, violation_by_type

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'
SEQ_BLUE = [[0.0, "#eef5fd"], [0.2, "#cde2fb"], [0.4, "#86b6ef"],
           [0.6, "#3987e5"], [0.8, "#1c5cab"], [1.0, "#0d366b"]]
print("repo root:", ROOT)

## 1. What to Observe — Violation Amounts in the Root LP Relaxation Solution

`_RootLPViolation` (`Eventhdlr`) fires at `FIRSTLPSOLVED` (the first LP solve = root). After assembling the LP relaxation solution as a `Sol`, it measures the following for each `NlRow` (nonlinear relaxation row, which retains the original constraint name) held by the model:

- `getNlRowSolFeasibility(nlrow, sol)` — If negative, it is a violation; its absolute value is the violation amount
- `getNlRowSolActivity(nlrow, sol)` — Left-hand side activity value (used for normalization)

Since the scale varies drastically across constraints, we use **Relative Violation = Violation Amount / (|Activity Value| + 1)** as the primary metric.
`violation_by_type` parses constraint names from the `ctype_entity` format (e.g., `demand_J5`) into `ctype`/`entity`, and aggregates the mean, max, and count by type.

In [ ]:
vdf = collect_root_violations(sp.build_model())
print(f"収集した非線形制約行: {len(vdf)}本")
vdf[["constraint", "ctype", "entity", "activity", "rel_violation"]].sort_values(
    "rel_violation", ascending=False).head(8)

## 2. Visualization — Heatmap of Type × Entity

Rows = Constraint type (descending order of mean relative violation), Columns = Job/Machine. Darker colors indicate looser relaxations.

In [ ]:
by_type = violation_by_type(vdf)
type_order = by_type["ctype"].tolist()
ent_order = sorted(vdf["entity"].unique(), key=lambda e: (e[0] != "J", e))
piv = (vdf.pivot_table(index="ctype", columns="entity", values="rel_violation", aggfunc="max")
      .reindex(index=type_order, columns=ent_order))
z = piv.values
text = [["" if v != v else f"{v:.2f}" for v in row] for row in z]

fig = go.Figure(go.Heatmap(
    z=z, x=ent_order, y=type_order, colorscale=SEQ_BLUE, zmin=0,
    text=text, texttemplate="%{text}", textfont=dict(size=10),
    colorbar=dict(title=dict(text="相対違反", side="right"), thickness=12,
                 tickfont=dict(color=C["muted"], size=10)),
    hovertemplate="%{y}_%{x}<br>相対違反 %{z:.3f}<extra></extra>", xgap=2, ygap=2))
fig.update_layout(
    title=dict(text="ルートLP緩和解の非線形制約違反(相対)— 濃いほど緩和が緩い",
              font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    xaxis=dict(title="ジョブ / マシン", side="top", tickfont=dict(color=C["ink2"])),
    yaxis=dict(autorange="reversed", tickfont=dict(color=C["ink2"])),
    margin=dict(l=90, r=20, t=64, b=16), height=420)
show(fig)

In [ ]:
g = by_type.iloc[::-1]
fig = go.Figure(go.Bar(
    x=g["mean_rel"], y=g["ctype"], orientation="h", marker=dict(color=C["s1"]),
    text=[f"{v:.2f}" for v in g["mean_rel"]], textposition="outside",
    customdata=g[["max_rel", "n"]],
    hovertemplate="%{y}<br>平均相対違反 %{x:.3f}<br>最大 %{customdata[0]:.3f} / %{customdata[1]}本<extra></extra>"))
fig.update_layout(
    title=dict(text="制約タイプ別の平均相対違反(支配的ボトルネックの特定)",
              font=dict(color=C["ink"], size=14, family=FONT), x=0.01),
    paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12),
    xaxis=dict(title="平均相対違反", gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"]), zeroline=False),
    yaxis=dict(tickfont=dict(color=C["ink2"])),
    margin=dict(l=90, r=48, t=44, b=40), height=300)
show(fig)

## 3. Interpretation — Correspondence with Diagnostic Rules

`bottleneck_type`/`bottleneck_rel_viol` (aggregated values used by the diagnosis) directly correspond to the top of this ranking.
The `weak_relaxation` rule fires when "Relative Violation $\ge 0.5$ **AND** Spatial Branching Contribution $\ge 0.3$"
(For the spatial branching contribution, refer to [Notebook 1: McCormick + Spatial Branching Tree](01_mccormick_spatial_tree.ipynb)).

In [ ]:
top = by_type.iloc[0]
print(f"支配的ボトルネック: {top['ctype']}  平均相対違反 {top['mean_rel']:.2f}")

report = mk.analyze(sp.build_model, name="plant-violation", time_limit=10)
print(report.summary())
print("report.metrics の bottleneck_type:", report.metrics.get("bottleneck_type"),
     " / bottleneck_rel_viol:", f"{report.metrics.get('bottleneck_rel_viol', 0):.2f}")

You can confirm that the top type in the heatmap matches `report.metrics["bottleneck_type"]`. **energy** (triple product $n\cdot s\cdot(T-T_0)$, directly linked to the objective function) and **conversion** (nested $\exp$) are prominently loose, while **arrhenius** (single-variable `exp`), **jobtime**, and **tmax** remain tight.
—— The tendency that McCormick-style relaxations become looser for terms involving products of multiple variables is evident at a glance by placing multiple types side-by-side (a contrast not visible in the `district_heating` version, which has only one type).

## Summary

- The violation heatmap visualizes "how far the root LP deviates from the true constraints" by constraint type × entity, and serves as the primary raw information for the `weak_relaxation` diagnosis itself.
- Normalization (`/(|Activity Value|+1)`) is essential, otherwise differences are buried under drastic scale disparities (e.g., energy is ~1e3).
- Once the dominant bottleneck is identified, the application targets for [Exact Linearization of Integer × Continuous](../improve/01_linearize_product.ipynb) or [PWL-SOS2](../improve/02_pwl_sos2.ipynb) can be narrowed down.

Related: [Playbook 0. Diagnosis Itself](../../playbook/00-diagnose.md) /
API [`minlpkit.collectors`](../../api/pipeline.md).